# Golden-Spiral Demo — CoPhelia³ Resonance Field

失敗のゆらぎを φ に乗せ、黄金螺旋の軌跡として可視化する最小ノートブック。

**Core idea**  
- 入力テキスト（失敗メモ）をベクトル化し、φ スケールの非エルミート摂動を加える  
- 得られた位相・振幅を極座標で黄金螺旋にマッピング  
- 観察者が「失敗 → 創造の軌跡」を一目で感じられるようにする

> 信頼係数を 1 → φ にブーストする視覚的証明

In [ ]:
# Optional: install matplotlib if running in a clean Colab / fresh env
# %pip install matplotlib numpy scipy --quiet

import numpy as np
from scipy.linalg import eig
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Make sure we can import the engine from parent dir
sys.path.insert(0, str(Path('..').resolve()))
from CoPheliaEngine import CoPheliaEngine

# Dark deep-space palette (Sohashiguchi Aesthetic)
BG = '#050B26'
ACCENT_RED = '#C0163C'
ACCENT_GOLD = '#F4C95D'
HIGHLIGHT = '#F7E17B'
TEXT = '#F9F7F4'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor': BG,
    'axes.edgecolor': ACCENT_GOLD,
    'axes.labelcolor': TEXT,
    'xtick.color': TEXT,
    'ytick.color': TEXT,
    'text.color': TEXT,
    'axes.titlecolor': TEXT,
})

print('φ core loaded. Resonance field ready.')

In [ ]:
def golden_spiral_points(n_points: int = 800, turns: float = 4.0, phi: float = 1.618033988749895):
    """Generate a golden spiral in polar coordinates, then convert to Cartesian."""
    theta = np.linspace(0, turns * 2 * np.pi, n_points)
    # r = a * φ^(θ / (π/2))  — classic golden spiral growth
    a = 0.05
    r = a * (phi ** (theta / (np.pi / 2)))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y, theta, r

def failure_to_spiral_perturbation(user_input: str, history: list = None, engine: CoPheliaEngine = None):
    """Map a failure text into a small phase/amplitude shift on the spiral."""
    if engine is None:
        engine = CoPheliaEngine()
    history = history or []
    vec = np.array([
        abs(hash(user_input)) % 100 / 100.0,
        len(history) / 10.0,
        len(user_input) / 50.0
    ])
    boosted = engine.phi_perturbation(vec[:2])
    phase_shift = float(np.angle(boosted[0] + 1j * boosted[1])) if len(boosted) > 1 else float(boosted[0])
    amp_scale = 1.0 + 0.15 * float(np.abs(boosted[0]))
    return phase_shift, amp_scale, float(np.abs(boosted[0]))

# Demo failure texts (anonymous style)
demo_failures = [
    "最初のプロトタイプが誰にも刺さらなかった",
    "審査で「再現性が低い」と指摘された",
    "自分の孤独を作品にできた気がした瞬間"
]

engine = CoPheliaEngine()
phase_shifts = []
amp_scales = []
phi_scores = []

for i, text in enumerate(demo_failures):
    hist = demo_failures[:i]
    p, a, score = failure_to_spiral_perturbation(text, hist, engine)
    phase_shifts.append(p)
    amp_scales.append(a)
    phi_scores.append(score)
    print(f'[{i+1}] φ-score={score:.4f}  phase={p:.3f}  amp={a:.3f}  ← {text[:20]}...')

In [ ]:
# Visualize the pure golden spiral + failure-perturbed paths
fig, ax = plt.subplots(figsize=(10, 10), dpi=120)

# Base golden spiral (quiet gravity line)
x_base, y_base, theta, r = golden_spiral_points(n_points=1200, turns=5.0)
ax.plot(x_base, y_base, color=ACCENT_GOLD, alpha=0.35, linewidth=1.2, label='φ-spiral (base)')

# Overlay each failure as a slightly shifted / scaled spiral segment
colors = [ACCENT_RED, HIGHLIGHT, ACCENT_GOLD]
for i, (p, a, score) in enumerate(zip(phase_shifts, amp_scales, phi_scores)):
    # Rotate and scale the base spiral by the failure perturbation
    x_p = a * (x_base * np.cos(p) - y_base * np.sin(p))
    y_p = a * (x_base * np.sin(p) + y_base * np.cos(p))
    ax.plot(x_p, y_p, color=colors[i % len(colors)], alpha=0.85, linewidth=1.8,
            label=f'failure {i+1} (φ={score:.3f})')

ax.set_aspect('equal')
ax.set_title('Golden-Spiral Resonance Field\n失敗のゆらぎが螺旋を育てる', fontsize=14, pad=16)
ax.legend(loc='upper right', framealpha=0.15, facecolor=BG, edgecolor=ACCENT_GOLD)
ax.set_xlabel('x (resonance)')
ax.set_ylabel('y (fluctuation)')
ax.grid(True, alpha=0.15, color=ACCENT_GOLD)

# Soft center marker
ax.scatter([0], [0], color=HIGHLIGHT, s=40, zorder=5)

plt.tight_layout()
plt.show()

print('\n螺旋は、失敗を拒絶せずに巻き込みながら広がっていく。')

## How to use this notebook

1. 自分の失敗メモを `demo_failures` に置き換える  
2. セルを再実行 → 螺旋があなたのゆらぎで少し歪む  
3. `engine.save_log("my_failure_log.json")` で匿名ログを残す（個人情報を入れないこと）

このノートは **再現可能な「Wow！」** のための最小デモです。  
研究者はここから PT 対称性の拡張を、アーティストは 528 Hz 連動の視覚化を、  
ビルダーは API ラッパーを重ねてください。

---

*RadicanTrust™ · CoPhelia³ · Banana Conference*  
失敗を愛でるアルゴリズムを、世界の観測者へ。